# Coffee & Health — Análise Exploratória de Dados

**UC:** Análise de Dados e Big Data · Projeto A3 · 2026/2

| Nome | RA |
|---|---|
| | |
| | |

## 1. Introdução

Contexto, apresentação do dataset e objetivos do trabalho.

> Os dados são **sintéticos**: os resultados mostram associações, não causa e efeito nem evidência clínica.

## 2. Bibliotecas e configuração

| Biblioteca | Uso no projeto |
|---|---|
| `pandas` | Leitura do CSV e manipulação dos dados em DataFrames |
| `matplotlib` | Base dos gráficos |
| `seaborn` | Gráficos estatísticos sobre o matplotlib |
| `python-dotenv` | Leitura dos valores fixos guardados no arquivo `.env` |
| `os` e `pathlib` | Variáveis de ambiente e caminhos de arquivo |

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv

### Configuração

Os valores fixos do projeto ficam no arquivo `.env` (veja `.env.example`). O valor ao lado de cada variável só é usado quando o `.env` não existe.

In [ ]:
# Raiz do repositório: o notebook pode ser executado a partir de notebooks/ ou da raiz
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(RAIZ / ".env")

DATA_PATH = RAIZ / os.getenv("DATA_PATH", "data/synthetic_coffee_health_10000.csv")
RANDOM_STATE = int(os.getenv("RANDOM_STATE", "42"))
TEST_SIZE = float(os.getenv("TEST_SIZE", "0.2"))
TOP_N_PAISES = int(os.getenv("TOP_N_PAISES", "5"))
FIG_LARGURA = float(os.getenv("FIG_LARGURA", "10"))
FIG_ALTURA = float(os.getenv("FIG_ALTURA", "5"))
FAIXAS_ETARIAS = [int(limite) for limite in os.getenv("FAIXAS_ETARIAS", "18,30,45,81").split(",")]
K_ANONIMATO = int(os.getenv("K_ANONIMATO", "5"))

# Aparência padrão dos gráficos e das tabelas
sns.set_theme(style="whitegrid", rc={"figure.figsize": (FIG_LARGURA, FIG_ALTURA)})
pd.set_option("display.max_columns", None)

## 3. Carga dos dados

O CSV original nunca é alterado: `df_original` guarda os dados como foram lidos e `df` é a cópia usada nas transformações.

> Na leitura, o pandas converte o texto `None` da coluna `Health_Issues` em valor ausente (`NaN`). Esses registros são tratados na Q2.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"CSV não encontrado em {DATA_PATH}. Confira DATA_PATH no arquivo .env.")

df_original = pd.read_csv(DATA_PATH)
df = df_original.copy()

print(f"Arquivo carregado: {DATA_PATH.name}")
print(f"{df.shape[0]} registros e {df.shape[1]} colunas")

## 4. Q1 — Conhecendo o dataset

Quantos registros e colunas existem, quais são os tipos de dados, as cinco primeiras linhas e o resumo estatístico das colunas numéricas.

### Registros e colunas

In [ ]:
n_registros, n_colunas = df.shape

print(f"Registros: {n_registros}")
print(f"Colunas: {n_colunas}")

### Tipos de dados

`int64` são números inteiros, `float64` são números decimais e `str` (ou `object`, em versões mais antigas do pandas) são textos.

In [ ]:
df.dtypes.to_frame("Tipo no pandas")

In [ ]:
df.dtypes.astype(str).value_counts().to_frame("Quantidade de colunas")

### Cinco primeiras linhas

In [ ]:
df.head()

### Resumo estatístico das colunas numéricas

O `describe()` considera todas as colunas com tipo numérico. Três delas não são medidas:

- `ID` é só um identificador, então média e desvio padrão não têm significado;
- `Smoking` e `Alcohol_Consumption` são indicadores `0`/`1`, e a média equivale à proporção de registros com valor `1`.

In [ ]:
df.describe()

### Classificação das variáveis

O tipo no pandas indica como o dado está armazenado, não o que ele representa. A tabela abaixo classifica cada coluna pelo tipo de dado estatístico:

- **Numérica contínua:** pode assumir valores decimais dentro de um intervalo;
- **Numérica discreta:** valores inteiros, resultado de contagem ou registrados sem casas decimais;
- **Categórica nominal:** categorias sem ordem;
- **Categórica ordinal:** categorias com ordem (`Poor` < `Fair` < `Good` < `Excellent`; `Low` < `Medium` < `High`; `Mild` < `Moderate` < `Severe`);
- **Binária:** só dois valores, `0` e `1`;
- **Identificador:** distingue os registros e não entra nas análises.

In [ ]:
tipos_de_dado = {
    "ID": "Identificador",
    "Age": "Numérica discreta",
    "Gender": "Categórica nominal",
    "Country": "Categórica nominal",
    "Coffee_Intake": "Numérica contínua",
    "Caffeine_mg": "Numérica contínua",
    "Sleep_Hours": "Numérica contínua",
    "Sleep_Quality": "Categórica ordinal",
    "BMI": "Numérica contínua",
    "Heart_Rate": "Numérica discreta",
    "Stress_Level": "Categórica ordinal",
    "Physical_Activity_Hours": "Numérica contínua",
    "Health_Issues": "Categórica ordinal",
    "Occupation": "Categórica nominal",
    "Smoking": "Binária",
    "Alcohol_Consumption": "Binária",
}

classificacao = pd.DataFrame({
    "Tipo no pandas": df.dtypes.astype(str),
    "Valores distintos": df.nunique(),
    "Classificação": pd.Series(tipos_de_dado),
})
classificacao

In [ ]:
classificacao["Classificação"].value_counts().to_frame("Quantidade de colunas")

### Resposta

- O dataset tem **10.000 registros** e **16 colunas**; cada registro representa uma pessoa.
- No pandas, 5 colunas são inteiras (`int64`), 5 são decimais (`float64`) e 6 são de texto.
- Pela classificação estatística, são 5 numéricas contínuas, 2 numéricas discretas, 3 categóricas nominais, 3 categóricas ordinais, 2 binárias e 1 identificador.
- A idade média é de cerca de 35 anos, o consumo médio de café é de 2,5 e a média de sono é de 6,6 horas.
- A média de `Smoking` (0,20) e de `Alcohol_Consumption` (0,30) mostra que cerca de 20% dos registros são de fumantes e 30% de pessoas que consomem álcool.
- `Health_Issues` já aparece com valores ausentes (`NaN`) nas primeiras linhas; eles são analisados na Q2.

## 5. Q2 — Qualidade dos dados e ETL

O dataset tem valores ausentes ou registros duplicados? Em quais colunas, em que quantidade e como tratá-los sem modificar o arquivo original.

O diagnóstico é feito sobre `df_original`, que guarda os dados exatamente como foram lidos do CSV.

### Valores ausentes

Um valor ausente indica que a informação não foi registrada. No pandas, ele aparece como `NaN`.

- `isnull()` devolve `True` em cada célula vazia e `False` nas preenchidas.
- `.sum()` soma esses valores por coluna (`True` vale 1), resultando na quantidade de ausentes.
- `.mean()` calcula a proporção de ausentes em cada coluna; multiplicada por 100, vira percentual.

> **Atenção:** no CSV, os ausentes de `Health_Issues` estão escritos como o texto `None`. O `read_csv` converte esse texto em `NaN` automaticamente. Ausência **não** significa "sem problemas de saúde"; significa apenas que o dado não foi informado.

In [ ]:
ausentes = pd.DataFrame({
    "Valores ausentes": df_original.isnull().sum(),
    "% do total": df_original.isnull().mean() * 100,
})
ausentes

### Registros duplicados

Três verificações: linhas inteiras repetidas, linhas repetidas quando o `ID` é ignorado (a mesma pessoa cadastrada duas vezes com identificadores diferentes) e identificadores repetidos.

In [ ]:
linhas_duplicadas = df_original.duplicated().sum()
linhas_duplicadas_sem_id = df_original.drop(columns="ID").duplicated().sum()
ids_repetidos = df_original["ID"].duplicated().sum()

print(f"Linhas duplicadas: {linhas_duplicadas}")
print(f"Linhas duplicadas ignorando o ID: {linhas_duplicadas_sem_id}")
print(f"IDs repetidos: {ids_repetidos}")

### Consistência das categorias

Confere se as colunas de texto têm só as categorias previstas no dicionário de dados, sem variações de grafia.

In [ ]:
colunas_categoricas = ["Gender", "Country", "Sleep_Quality", "Stress_Level", "Health_Issues", "Occupation"]

for coluna in colunas_categoricas:
    categorias = sorted(df_original[coluna].dropna().unique())
    print(f"{coluna} ({len(categorias)}): {categorias}")

### Fluxo ETL

```
┌──────────────┐      ┌────────────────────────────┐      ┌──────────────────┐
│   EXTRAÇÃO   │      │       TRANSFORMAÇÃO        │      │      CARGA       │
│              │ ───► │                            │ ───► │                  │
│  CSV → df_   │      │  cópia → duplicados →      │      │  df: DataFrame   │
│  original    │      │  ausentes → ordinais →     │      │  limpo, usado    │
│              │      │  anonimização              │      │  nas análises    │
└──────────────┘      └────────────────────────────┘      └──────────────────┘
```

| Fase | Passo | O que é feito |
|---|---|---|
| Extração | Leitura | `pd.read_csv` lê o arquivo para `df_original` (seção 3). O CSV nunca é regravado. |
| Transformação | 1. Cópia | `df_original.copy()`: todas as mudanças são feitas na cópia. |
| Transformação | 2. Duplicados | `drop_duplicates()` remove linhas repetidas, se existirem. |
| Transformação | 3. Ausentes | Os vazios de `Health_Issues` recebem o rótulo `Não informado`. |
| Transformação | 4. Ordinais | `Sleep_Quality`, `Stress_Level` e `Health_Issues` passam a ter as categorias na ordem correta. |
| Transformação | 5. Anonimização | A coluna `ID` é removida, porque identifica o registro e não é usada nas análises. |
| Carga | DataFrame limpo | O resultado fica em `df`, usado em todas as seções seguintes. |

### Transformação

In [ ]:
ROTULO_NAO_INFORMADO = "Não informado"
ORDEM_SLEEP_QUALITY = ["Poor", "Fair", "Good", "Excellent"]
ORDEM_STRESS_LEVEL = ["Low", "Medium", "High"]
ORDEM_HEALTH_ISSUES = ["Mild", "Moderate", "Severe"]

# 1. Cópia: o original fica preservado em df_original
df = df_original.copy()

# 2. Duplicados
df = df.drop_duplicates()

# 3. Ausentes: vazio não significa "sem problema de saúde"
df["Health_Issues"] = df["Health_Issues"].fillna(ROTULO_NAO_INFORMADO)

# 4. Ordinais: categorias na ordem correta
df["Sleep_Quality"] = pd.Categorical(df["Sleep_Quality"], categories=ORDEM_SLEEP_QUALITY, ordered=True)
df["Stress_Level"] = pd.Categorical(df["Stress_Level"], categories=ORDEM_STRESS_LEVEL, ordered=True)
df["Health_Issues"] = pd.Categorical(df["Health_Issues"], categories=ORDEM_HEALTH_ISSUES + [ROTULO_NAO_INFORMADO])

# 5. Anonimização: remove o identificador
df = df.drop(columns="ID").reset_index(drop=True)

### Carga

Conferência do DataFrame limpo e de que o original continua intacto.

In [ ]:
print(f"Original: {df_original.shape[0]} registros e {df_original.shape[1]} colunas")
print(f"Limpo:    {df.shape[0]} registros e {df.shape[1]} colunas")
print(f"Valores ausentes no DataFrame limpo: {df.isnull().sum().sum()}")
print(f"Valores ausentes no original: {df_original.isnull().sum().sum()}")

In [ ]:
df["Health_Issues"].value_counts().to_frame("Registros")

In [ ]:
df.head()

### Resposta

- **Valores ausentes:** só a coluna `Health_Issues` tem valores ausentes: **5.941 registros (59,41%)**. As outras 15 colunas estão completas.
- **Duplicados:** não há registros duplicados, nem linhas inteiras, nem linhas iguais com `ID` diferente, nem `ID` repetido.
- **Categorias:** as colunas de texto têm só as categorias previstas, sem variações de grafia.

**Como tratar sem modificar o arquivo original:** o CSV é apenas lido, e todas as mudanças são feitas em `df`, uma cópia de `df_original`. Para os vazios de `Health_Issues` há duas opções, usadas conforme a análise:

1. **Rotular como `Não informado`** (feito acima): mantém os 10.000 registros nas análises que não dependem dessa coluna.
2. **Filtrar só os registros preenchidos**: é o que a Q10 pede, com `df[df["Health_Issues"] != ROTULO_NAO_INFORMADO]`.

Não é adequado preencher os vazios com a categoria mais frequente (`Mild`) nem tratá-los como "sem problema de saúde". No CSV esses campos trazem o texto `None`, mas o dicionário de dados orienta a não presumir que ausência significa ausência de problema, e alterar quase 60% da coluna distorceria qualquer comparação.

> **Para as próximas questões:** `Sleep_Quality`, `Stress_Level` e `Health_Issues` agora são colunas categóricas. Ao agrupar por elas, use `groupby(..., observed=True)` para que só apareçam as categorias e combinações que existem nos dados.

## 6. Análise Exploratória (EDA)

### Q3 — Perfil das pessoas

In [ ]:
# TODO

### Q4 — Consumo de café

In [ ]:
# TODO

### Q5 — Café e cafeína

In [ ]:
# TODO

### Q6 — Consumo de café e sono

In [ ]:
# TODO

### Q7 — Sono e estresse

In [ ]:
# TODO

### Q8 — Classificação com Árvore de Decisão

In [ ]:
# TODO

### Q9 — Análise conjunta de consumo, sono e estresse

In [ ]:
# TODO

### Q10 — Possíveis associações com problemas de saúde

In [ ]:
# TODO

## 7. LGPD e ética

Quais dados do dataset são pessoais ou sensíveis, quais riscos éticos o tratamento dessas informações traz e como o projeto se adequa aos princípios da LGPD (Lei nº 13.709/2018).

> O dataset é **sintético**: nenhum registro corresponde a uma pessoa real. Mesmo assim, a análise trata os dados como se fossem reais, porque a mesma estrutura de colunas, coletada de pessoas de verdade, estaria sujeita à LGPD.

### Classificação dos dados segundo a LGPD

- **Dado pessoal** (art. 5º, I): informação relacionada a pessoa natural identificada ou identificável.
- **Dado pessoal sensível** (art. 5º, II): entre outros, dado referente à saúde. Exige uma base legal mais restrita (art. 11).
- **Identificador direto:** aponta para um registro específico (`ID`).
- **Quase-identificador:** sozinho não identifica ninguém, mas combinado com outros pode identificar (`Age`, `Gender`, `Country`, `Occupation`).

In [ ]:
QUASE_IDENTIFICADORES = ["Age", "Gender", "Country", "Occupation"]

categorias_lgpd = {
    "ID": "Identificador direto",
    "Age": "Dado pessoal (quase-identificador)",
    "Gender": "Dado pessoal (quase-identificador)",
    "Country": "Dado pessoal (quase-identificador)",
    "Occupation": "Dado pessoal (quase-identificador)",
    "Coffee_Intake": "Dado pessoal (hábito)",
    "Caffeine_mg": "Dado pessoal (hábito)",
    "Physical_Activity_Hours": "Dado pessoal (hábito)",
    "Sleep_Hours": "Dado sensível (saúde)",
    "Sleep_Quality": "Dado sensível (saúde)",
    "Stress_Level": "Dado sensível (saúde)",
    "BMI": "Dado sensível (saúde)",
    "Heart_Rate": "Dado sensível (saúde)",
    "Health_Issues": "Dado sensível (saúde)",
    "Smoking": "Dado sensível (saúde)",
    "Alcohol_Consumption": "Dado sensível (saúde)",
}

classificacao_lgpd = pd.Series(categorias_lgpd).to_frame("Categoria LGPD")
classificacao_lgpd

In [ ]:
classificacao_lgpd["Categoria LGPD"].value_counts().to_frame("Quantidade de colunas")

### Risco de reidentificação

Remover o `ID` (feito no ETL) não basta para anonimizar. Quem conhece a idade, o gênero, o país e a ocupação de alguém pode encontrar o registro dessa pessoa se a combinação for rara, e com isso descobrir os dados de saúde dela.

A medida usada aqui é o **k-anonimato**: cada combinação de quase-identificadores deve aparecer em pelo menos `K_ANONIMATO` registros. Quanto menor o grupo, mais fácil reidentificar; um grupo de tamanho 1 identifica a pessoa.

In [ ]:
def medir_risco(dados, colunas, k=K_ANONIMATO):
    """Agrupa os registros pelas colunas informadas e mede quantos ficam em grupos menores que k."""
    tamanhos = dados.groupby(colunas, observed=True).size()
    em_risco = tamanhos[tamanhos < k].sum()
    # dtype object mantém as contagens como inteiros ao lado do percentual
    return pd.Series(dtype=object, data={
        "Registros": len(dados),
        "Combinações distintas": len(tamanhos),
        "Registros únicos na combinação": (tamanhos == 1).sum(),
        f"Registros em grupos com menos de {k}": em_risco,
        "% em risco": round(em_risco / len(dados) * 100, 2),
    })


medir_risco(df, QUASE_IDENTIFICADORES).to_frame("Idade exata")

### Medidas de proteção: generalização e supressão

Duas técnicas de anonimização aplicadas a uma versão do DataFrame destinada a compartilhamento:

1. **Generalização:** a idade exata é substituída por faixas etárias (limites definidos em `FAIXAS_ETARIAS`, no `.env`).
2. **Supressão:** os registros que, mesmo com as faixas, continuam em grupos com menos de `K_ANONIMATO` pessoas são retirados.

As análises do notebook continuam usando `df`, porque questões como a Q3 e a Q8 precisam da idade exata. `df_compartilhavel` é a versão que poderia sair do ambiente de análise.

> **Por que a última faixa vai de 45 a 80 anos:** há poucos registros acima de 60 anos (185). Com uma faixa separada de 60 a 80, todos eles caíam em grupos pequenos e a supressão excluía a faixa inteira, deixando os idosos fora dos dados compartilhados. Juntar as faixas reduz a precisão da idade, mas evita apagar um grupo da população.

In [ ]:
rotulos_faixas = [f"{inicio}–{fim - 1}" for inicio, fim in zip(FAIXAS_ETARIAS[:-1], FAIXAS_ETARIAS[1:])]
quase_identificadores_generalizados = ["Faixa_Etaria", "Gender", "Country", "Occupation"]

# Generalização: idade exata → faixa etária
df_generalizado = df.copy()
df_generalizado["Faixa_Etaria"] = pd.cut(
    df_generalizado["Age"], bins=FAIXAS_ETARIAS, labels=rotulos_faixas, right=False
)
df_generalizado = df_generalizado.drop(columns="Age")

# Supressão: retira os registros que continuam em grupos menores que K_ANONIMATO
tamanho_do_grupo = df_generalizado.groupby(
    quase_identificadores_generalizados, observed=True
)["Gender"].transform("size")
df_compartilhavel = df_generalizado[tamanho_do_grupo >= K_ANONIMATO].reset_index(drop=True)

pd.DataFrame({
    "Idade exata": medir_risco(df, QUASE_IDENTIFICADORES),
    "Faixa etária": medir_risco(df_generalizado, quase_identificadores_generalizados),
    "Faixa etária + supressão": medir_risco(df_compartilhavel, quase_identificadores_generalizados),
})

In [ ]:
df_compartilhavel["Faixa_Etaria"].value_counts(sort=False).to_frame("Registros")

### Riscos éticos

| Risco | Como aconteceria |
|---|---|
| **Reidentificação** | Cruzar idade, gênero, país e ocupação com outra fonte (rede social, cadastro de empresa) e chegar à pessoa e aos seus dados de saúde. |
| **Discriminação** | Seguradoras, planos de saúde ou empregadores usarem `BMI`, `Health_Issues`, `Smoking` ou `Alcohol_Consumption` para negar cobertura, cobrar mais ou recusar candidatos. |
| **Desvio de finalidade** | Dados coletados para um estudo sobre café e sono serem usados para publicidade ou perfilamento de consumo. |
| **Vazamento** | Acesso indevido ao arquivo expor dados sensíveis de milhares de pessoas. |
| **Conclusões causais indevidas** | Apresentar associações como causa ("café causa problemas de saúde") e estigmatizar grupos, como fumantes ou certas ocupações. |
| **Viés em modelos** | Um modelo como o da Q8 ser usado para decidir algo sobre uma pessoa, reproduzindo distorções da amostra (por exemplo, o grupo `Other` em `Gender` tem poucos registros). |

### Adequação aos princípios da LGPD

**Base legal.** Por envolver dados de saúde, um tratamento real exigiria uma das hipóteses do art. 11, como o consentimento específico e destacado do titular ou a realização de estudos por órgão de pesquisa, garantida, sempre que possível, a anonimização dos dados sensíveis. Dados anonimizados de forma irreversível deixam de ser considerados dados pessoais (art. 12).

| Princípio (art. 6º) | Como o projeto aplica |
|---|---|
| Finalidade | Os dados são usados só para a análise exploratória e acadêmica descrita no enunciado. |
| Adequação | As análises têm relação direta com o tema do estudo (café, sono, estresse e saúde). |
| Necessidade | Minimização: o `ID` é removido no ETL e a idade é generalizada na versão compartilhável. |
| Livre acesso e transparência | O README e o notebook documentam quais dados são usados, como e para quê. |
| Qualidade dos dados | O ETL verifica ausentes, duplicados e categorias sem alterar o arquivo original. |
| Segurança e prevenção | Repositório privado, acesso restrito aos integrantes e compartilhamento apenas de `df_compartilhavel` (art. 46). |
| Não discriminação | Os resultados são apresentados como associações em dados sintéticos, nunca como critério para decisões sobre pessoas. |
| Responsabilização | As decisões de tratamento ficam registradas no código, no histórico do Git e nos pull requests. |

### Resposta

- **Dados sensíveis:** 8 das 16 colunas são dados referentes à saúde (`Sleep_Hours`, `Sleep_Quality`, `Stress_Level`, `BMI`, `Heart_Rate`, `Health_Issues`, `Smoking` e `Alcohol_Consumption`). `ID` é identificador direto, e `Age`, `Gender`, `Country` e `Occupation` são quase-identificadores.
- **Remover o ID não anonimiza:** com a idade exata, 3.084 registros são únicos na combinação de quase-identificadores e 8.910 (89,1%) ficam em grupos com menos de 5 pessoas.
- **Generalização e supressão funcionam:** com as faixas etárias (18–29, 30–44 e 45–80), os registros em risco caem para 237 (2,37%); retirando esses registros, todos os grupos passam a ter pelo menos 5 pessoas e 9.763 registros (97,6%) são preservados, sem excluir nenhuma faixa etária.
- **Principais riscos éticos:** reidentificação, discriminação por seguradoras ou empregadores, desvio de finalidade, vazamento e conclusões causais indevidas.
- **Adequação à LGPD:** base legal do art. 11 para dados de saúde, anonimização sempre que possível (art. 12), minimização, segurança no armazenamento e transparência sobre o tratamento. Como o dataset é sintético, essas medidas demonstram como o projeto se comportaria com dados reais.

## 8. Hipóteses

Cada hipótese: enunciado → análise → resultado → conclusão.

### Hipótese 1

In [ ]:
# TODO

### Hipótese 2

In [ ]:
# TODO

### Hipótese 3

In [ ]:
# TODO

### Hipótese 4

In [ ]:
# TODO

## 9. Conclusões